In [1]:
# Step 1: Load data and create forecasting features

from pathlib import Path
import pandas as pd
import numpy as np

# Locate the project root
current = Path.cwd()

project_root = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed" / "perishable_inventory.csv").exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Cannot find data/processed/perishable_inventory.csv"
    )

data_path = (
    project_root
    / "data"
    / "processed"
    / "perishable_inventory.csv"
)

df = pd.read_csv(data_path)
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Validate required columns
required_columns = {"product_id", "date", "units_sold"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

# Sort chronologically within each product
df = (
    df.dropna(subset=["product_id", "date"])
    .sort_values(["product_id", "date"])
    .reset_index(drop=True)
)

# Create a separate feature dataset
df_features = df.copy()

# Calendar features
df_features["day_of_week"] = df_features["date"].dt.dayofweek
df_features["day_of_month"] = df_features["date"].dt.day
df_features["month"] = df_features["date"].dt.month

df_features["is_weekend"] = (
    df_features["day_of_week"] >= 5
).astype(int)

# Historical sales features
grouped_sales = df_features.groupby("product_id")["units_sold"]

df_features["sales_lag_1"] = grouped_sales.shift(1)
df_features["sales_lag_7"] = grouped_sales.shift(7)

# Average of the previous 7 observations, excluding the current day
df_features["sales_avg_7d"] = grouped_sales.transform(
    lambda s: s.shift(1).rolling(window=7, min_periods=1).mean()
)

print("Feature engineering completed successfully!")
print("Original dataset shape:", df.shape)
print("Feature dataset shape:", df_features.shape)

display(df_features.head(10))

Feature engineering completed successfully!
Original dataset shape: (1800, 12)
Feature dataset shape: (1800, 19)


,product_id,date,category,opening_stock,units_received,units_sold,unit_price,discount_pct,days_to_expiry,temperature_c,waste_units,stockout_flag,day_of_week,day_of_month,month,is_weekend,sales_lag_1,sales_lag_7,sales_avg_7d
0,1001,2026-04-01,Dairy,27,0,6,32.0,0,1,4.1,0,0,2,1,4,0,NaN,NaN,NaN
1,1001,2026-04-02,Dairy,30,12,5,32.0,0,5,3.7,0,0,3,2,4,0,6.0,NaN,6.000000
2,1001,2026-04-03,Dairy,38,0,14,32.0,0,2,4.6,0,0,4,3,4,0,5.0,NaN,5.500000
3,1001,2026-04-04,Dairy,49,0,11,32.0,0,0,5.7,0,0,5,4,4,1,14.0,NaN,8.333333
4,1001,2026-04-05,Dairy,19,13,12,32.0,0,0,3.0,0,0,6,5,4,1,11.0,NaN,9.000000
5,1001,2026-04-06,Dairy,27,0,13,32.0,0,1,4.1,0,0,0,6,4,0,12.0,NaN,9.600000
6,1001,2026-04-07,Dairy,32,0,14,32.0,0,0,6.2,0,0,1,7,4,0,13.0,NaN,10.166667
7,1001,2026-04-08,Dairy,35,0,14,32.0,0,0,5.1,0,0,2,8,4,0,14.0,6.0,10.714286
8,1001,2026-04-09,Dairy,22,0,8,32.0,0,0,2.6,0,0,3,9,4,0,14.0,5.0,11.857143
9,1001,2026-04-10,Dairy,27,0,11,32.0,0,1,3.5,0,0,4,10,4,0,8.0,14.0,12.285714


In [2]:
# Step 2: Check missing values in engineered features

print("MISSING VALUES IN ENGINEERED DATA")
print("-" * 45)

missing_summary = df_features.isnull().sum()
display(
    missing_summary[missing_summary > 0]
    .rename("missing_count")
    .to_frame()
)

print("\nMISSING LAG VALUES")
print("-" * 45)

lag_columns = [
    "sales_lag_1",
    "sales_lag_7",
    "sales_avg_7d",
]

for column in lag_columns:
    print(f"{column}: {df_features[column].isna().sum()}")

# Keep rows that have the historical sales needed for forecasting
df_model = df_features.dropna(
    subset=["sales_lag_1", "sales_lag_7"]
).copy()

print("\nFEATURE DATASET SUMMARY")
print("-" * 45)
print("Original rows:", len(df_features))
print("Rows available for modeling:", len(df_model))
print("Rows removed due to missing lags:", len(df_features) - len(df_model))
print("Unique products:", df_model["product_id"].nunique())

display(df_model.head())

MISSING VALUES IN ENGINEERED DATA
---------------------------------------------


,missing_count
sales_lag_1,10
sales_lag_7,70
sales_avg_7d,10



MISSING LAG VALUES
---------------------------------------------
sales_lag_1: 10
sales_lag_7: 70
sales_avg_7d: 10

FEATURE DATASET SUMMARY
---------------------------------------------
Original rows: 1800
Rows available for modeling: 1730
Rows removed due to missing lags: 70
Unique products: 10


,product_id,date,category,opening_stock,units_received,units_sold,unit_price,discount_pct,days_to_expiry,temperature_c,waste_units,stockout_flag,day_of_week,day_of_month,month,is_weekend,sales_lag_1,sales_lag_7,sales_avg_7d
7,1001,2026-04-08,Dairy,35,0,14,32.0,0,0,5.1,0,0,2,8,4,0,14.0,6.0,10.714286
8,1001,2026-04-09,Dairy,22,0,8,32.0,0,0,2.6,0,0,3,9,4,0,14.0,5.0,11.857143
9,1001,2026-04-10,Dairy,27,0,11,32.0,0,1,3.5,0,0,4,10,4,0,8.0,14.0,12.285714
10,1001,2026-04-11,Dairy,21,7,16,32.0,0,5,4.7,0,0,5,11,4,1,11.0,11.0,11.857143
11,1001,2026-04-12,Dairy,34,0,11,32.0,0,3,1.8,0,0,6,12,4,1,16.0,12.0,12.571429


In [3]:
# Step 3: Save the engineered feature dataset

output_dir = project_root / "data" / "processed" / "features"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "forecast_features.csv"

df_model.to_csv(output_path, index=False)

print("Feature dataset saved successfully!")
print("Saved location:", output_path)
print("Final shape:", df_model.shape)

Feature dataset saved successfully!
Saved location: c:\Users\hp\OneDrive\Desktop\web_scraping_folder\wasteAI_project\WasteWise-AI\data\processed\features\forecast_features.csv
Final shape: (1730, 19)
